In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

In [16]:
train_path = "/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/train"
val_path   = "/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/train"
test_path  = "/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test"

In [17]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 512 images belonging to 4 classes.
Found 128 images belonging to 4 classes.
Found 160 images belonging to 4 classes.


In [18]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

In [19]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [20]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20
)

Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 34s 888ms/step - accuracy: 0.2513 - loss: 1.6046 - val_accuracy: 0.2500 - val_loss: 1.3951
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 460ms/step - accuracy: 0.2569 - loss: 1.5413 - val_accuracy: 0.2500 - val_loss: 1.3918
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 484ms/step - accuracy: 0.2664 - loss: 1.4771 - val_accuracy: 0.2500 - val_loss: 1.3902
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 482ms/step - accuracy: 0.2382 - loss: 1.4958 - val_accuracy: 0.2500 - val_loss: 1.3878
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 495ms/step - accuracy: 0.2406 - loss: 1.5138 - val_accuracy: 0.2500 - val_loss: 1.3877
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 497ms/step - accuracy: 0.2536 - loss: 1.4809 - val_accuracy: 0.2500 - val_loss: 1.3890
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 489ms/step - accuracy: 0.2310 - loss: 1.5148 - val_accuracy: 0.2500 - val_loss: 1.3879
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - accuracy: 0.2797 - loss: 1.4534 - val_accuracy: 0

In [21]:
test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.1632 - loss: 1.3938 
Test Accuracy: 0.25
Test Loss: 1.3857189416885376


In [22]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

In [23]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [24]:
history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.2222 - loss: 2.0630 - val_accuracy: 0.2500 - val_loss: 1.3941
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 498ms/step - accuracy: 0.2451 - loss: 1.9896 - val_accuracy: 0.2500 - val_loss: 1.4145
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - accuracy: 0.2831 - loss: 1.8308 - val_accuracy: 0.2500 - val_loss: 1.4395
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 495ms/step - accuracy: 0.2632 - loss: 1.8613 - val_accuracy: 0.2500 - val_loss: 1.4714
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 492ms/step - accuracy: 0.2700 - loss: 1.7777 - val_accuracy: 0.2500 - val_loss: 1.5003
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - accuracy: 0.3072 - loss: 1.7128 - val_accuracy: 0.2500 - val_loss: 1.5423
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 488ms/step - accuracy: 0.2122 - loss: 1.9325 - val_accuracy: 0.2500 - val_loss: 1.5793
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 490ms/step - accuracy: 0.2420 - loss: 1.7943 - val_accuracy: 0.25

In [25]:
test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.1632 - loss: 1.8432 
Test Accuracy: 0.25
Test Loss: 1.6299245357513428


In [26]:
model.save("/kaggle/working/rice_classifier_finetuned_EfficientNetB0.keras")